Pada Notebook ini Fold Traing (p1) dan Foldd Testing (p2) sebagai berikt:

>P1= "12/06/2015 - 21/06/2015"


>p2= "22/06/2015 - 01/07/2015"

In [1]:
from neo4j import GraphDatabase, RoutingControl
import pandas as pd
from tqdm import tqdm
import math

URI = "neo4j://localhost:7687"
AUTH = ("neo4j", "password")

driver = GraphDatabase.driver(URI, auth=AUTH)
session = driver.session(database="system")

RAW_PATH = "../UTS/Local_data/events.csv"
OUT_DIR = "../UTS/Data"

# 2.1   

## 1. Filter data set 

In [2]:
RAW_PATH = "../UTS/Local_data/events.csv"
OUT_DIR = "../UTS/Data"

PERIODS = {
    "P1": ("2015-06-12", "2015-06-21"),   # 12/06/2015 - 21/06/2015
    "P2": ("2015-06-22", "2015-07-01"),   # 22/06/2015 - 01/07/2015
}

df = pd.read_csv(RAW_PATH, dtype={"transactionid": "string"})
df["datetime"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)  # zona waktu: UTC

result = {}
for name, (start, end) in PERIODS.items():
    lo = pd.Timestamp(start, tz="UTC")
    hi = pd.Timestamp(end, tz="UTC") + pd.Timedelta(days=1)  # tanggal akhir ikut penuh
    part = df[(df["datetime"] >= lo) & (df["datetime"] < hi)].copy()
    part["period"] = name
    part["datetime"] = part["datetime"].dt.strftime("%Y-%m-%d %H:%M:%S")
    part.to_csv(f"{OUT_DIR}/events_{name}.csv", index=False)
    result[name] = part

# --- pengecekan ---
for name, part in result.items():
    print(name, "| baris:", len(part),
          "| visitor:", part["visitorid"].nunique(),
          "| item:", part["itemid"].nunique())
    print(part["event"].value_counts().to_string(), "\n")

v1, v2 = set(result["P1"]["visitorid"]), set(result["P2"]["visitorid"])
i1, i2 = set(result["P1"]["itemid"]), set(result["P2"]["itemid"])
print("Visitor di kedua periode:", len(v1 & v2))
print("Item di kedua periode   :", len(i1 & i2))

P1 | baris: 188267 | visitor: 99659 | item: 59808
event
view           181674
addtocart        5002
transaction      1591 

P2 | baris: 206884 | visitor: 110012 | item: 64577
event
view           200049
addtocart        5053
transaction      1782 

Visitor di kedua periode: 4777
Item di kedua periode   : 31454


In [3]:
URI = "neo4j://localhost:7687"   
AUTH = ("neo4j", "password")
DATABASE = "neo4j"

OUT_DIR = "../UTS/Data"
FILES = {
    "P1": f"{OUT_DIR}/events_P1.csv",
    "P2": f"{OUT_DIR}/events_P2.csv",
}
BATCH_SIZE = 5000
RESET_DB = False   

In [4]:
def create_constraints(session):
    session.run(
        "CREATE CONSTRAINT visitor_id_unique IF NOT EXISTS "
        "FOR (v:Visitor) REQUIRE v.visitorId IS UNIQUE"
    ).consume()

    session.run(
        "CREATE CONSTRAINT item_id_unique IF NOT EXISTS "
        "FOR (i:Item) REQUIRE i.itemId IS UNIQUE"
    ).consume()

    session.run(
        "CREATE CONSTRAINT transaction_id_unique IF NOT EXISTS "
        "FOR (t:Transaction) REQUIRE t.transactionId IS UNIQUE"
    ).consume()

    session.run(
        "CREATE CONSTRAINT category_id_unique IF NOT EXISTS "
        "FOR (c:Category) REQUIRE c.categoryId IS UNIQUE"
    ).consume()


In [5]:
Q_VIEW = """
UNWIND $rows AS row
MERGE (v:Visitor {visitorId: row.visitorid})
MERGE (i:Item {itemId: row.itemid})
CREATE (v)-[:VIEWED {timestamp: row.timestamp, period: row.period}]->(i)
"""

Q_CART = """
UNWIND $rows AS row
MERGE (v:Visitor {visitorId: row.visitorid})
MERGE (i:Item {itemId: row.itemid})
CREATE (v)-[:ADDED_TO_CART {timestamp: row.timestamp, period: row.period}]->(i)
"""

Q_BUY = """
UNWIND $rows AS row
MERGE (v:Visitor {visitorId: row.visitorid})
MERGE (i:Item {itemId: row.itemid})
MERGE (t:Transaction {transactionId: row.transactionid})
CREATE (v)-[:BOUGHT {timestamp: row.timestamp, period: row.period,
                     transactionId: row.transactionid}]->(i)
MERGE (v)-[:MADE]->(t)
MERGE (t)-[:CONTAINS]->(i)
"""

Q_EXISTS = "MATCH ()-[r]->() WHERE r.period = $period RETURN count(r) AS n"



In [6]:
def import_batch(tx, records):
    views = [r for r in records if r["event"] == "view"]
    carts = [r for r in records if r["event"] == "addtocart"]
    buys = [r for r in records if r["event"] == "transaction" and r["transactionid"]]
    if views:
        tx.run(Q_VIEW, rows=views).consume()
    if carts:
        tx.run(Q_CART, rows=carts).consume()
    if buys:
        tx.run(Q_BUY, rows=buys).consume()


def count_rows(path):
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for _ in f) - 1   # kurangi baris header


def import_file(session, period, path):
    existing = session.execute_read(
        lambda tx: tx.run(Q_EXISTS, period=period).single()["n"]
    )
    if existing:
        raise RuntimeError(
            f"Periode {period} sudah ada di graf ({existing} relasi). "
            "Impor dibatalkan agar tidak terduplikasi."
        )

    total_rows = count_rows(path)
    total_chunks = -(-total_rows // BATCH_SIZE)   # pembulatan ke atas
    print(f"[{period}] membaca {path} ({total_rows} baris)")

    chunks = pd.read_csv(path, chunksize=BATCH_SIZE, dtype={"transactionid": "string"})
    skipped = 0
    for chunk in tqdm(chunks, total=total_chunks, desc=f"Impor {period}"):
        # Membersihkan data null/NaN pada transactionid
        chunk["transactionid"] = chunk["transactionid"].fillna("")
        skipped += int(((chunk["event"] == "transaction") & (chunk["transactionid"] == "")).sum())
        records = chunk.astype(object).to_dict(orient="records")  # tipe Python native
        session.execute_write(import_batch, records)

    print(f"[{period}] selesai. Baris transaksi tanpa id yang dilewati: {skipped}")


In [ ]:
driver = GraphDatabase.driver(URI, auth=AUTH)
try:
    driver.verify_connectivity()
    with driver.session(database=DATABASE) as session:
        if RESET_DB:
            session.run("MATCH (n) DETACH DELETE n").consume()
        create_constraints(session)
        for period, path in FILES.items():
            import_file(session, period, path)
    print("Proses impor berhasil.")
finally:
    driver.close()


RuntimeError: Periode P1 sudah ada di graf (188267 relasi). Impor dibatalkan agar tidak terduplikasi.